In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

SUPPORT_DIR = "/content/drive/MyDrive/zepto-data-ai-platform/support_assistant"
DOCS_DIR = f"{SUPPORT_DIR}/docs"

os.makedirs(DOCS_DIR, exist_ok=True)

print(SUPPORT_DIR)

Mounted at /content/drive
/content/drive/MyDrive/zepto-data-ai-platform/support_assistant


In [2]:
!pip install -q sentence-transformers chromadb langgraph fastapi uvicorn pydantic groq httpx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

In [3]:
try:
    import chromadb
    import sentence_transformers
    import langgraph
    import fastapi
    import uvicorn
    import pydantic

    print("chromadb")
    print("sentence-transformers")
    print("langgraph")
    print("fastapi")
    print("uvicorn")
    print("pydantic")
    print("\nALL REQUIRED PACKAGES ARE WORKING")

except Exception as e:
    print("ERROR:", e)

chromadb
sentence-transformers
langgraph
fastapi
uvicorn
pydantic

ALL REQUIRED PACKAGES ARE WORKING


In [5]:
documents = {
    "doc_01.txt": """Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.""",

    "doc_02.txt": """Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3–5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit. Personal care items that have been opened are non-returnable except in the case of a manufacturing defect. Return pickup, where required, is arranged free of cost by Zepto.""",

    "doc_03.txt": """Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limited-time deals 24 hours before they go live to Basic and Pass members). Membership can be cancelled at any time from account settings; cancelling stops the next billing cycle but does not refund the current membership period.""",

    "doc_04.txt": """Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the rider moves. If an order's status shows no movement for more than 20 minutes past its original estimated delivery time, customers should contact support directly rather than continue waiting, since this indicates a likely delivery issue.""",

    "doc_05.txt": """Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due to a Zepto-side issue (for example, rider unavailability), the order is auto-cancelled and fully refunded without any cancellation fee.""",

    "doc_06.txt": """If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the 'Report an Issue' button on the order page. Zepto ships a free replacement or issues a full refund for damaged, spoiled, or missing items without requiring the customer to return the original item, unless the order value exceeds INR 1000, in which case a photo of the issue must be submitted through the report form before a replacement or refund is processed.""",

    "doc_07.txt": """Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees. Gift card balance can be combined with one other payment method at checkout but cannot be combined with another gift card in the same transaction. Gift card balance cannot be redeemed for cash except where required by law.""",

    "doc_08.txt": """Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given the time-sensitive nature of quick commerce deliveries. Average in-app chat response time is under 2 minutes. Email support is also available for non-urgent queries and is answered within 24 hours on business days. Phone support is not offered."""
}

for filename, content in documents.items():
    with open(f"{DOCS_DIR}/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

print("8 policy documents created")
print(os.listdir(DOCS_DIR))

8 policy documents created
['doc_01.txt', 'doc_02.txt', 'doc_03.txt', 'doc_04.txt', 'doc_05.txt', 'doc_06.txt', 'doc_07.txt', 'doc_08.txt']


In [6]:
%cd /content/drive/MyDrive/zepto-data-ai-platform/support_assistant
!ls

/content/drive/MyDrive/zepto-data-ai-platform/support_assistant
docs  module3_builder.ipynb


In [7]:
!pip install -q sentence-transformers chromadb

In [8]:
%%writefile ingest.py

from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer

BASE_DIR = Path(__file__).parent
DOCS_DIR = BASE_DIR / "docs"
DB_DIR = BASE_DIR / "chroma_db"

# Embedding model required for the project
model = SentenceTransformer("all-MiniLM-L6-v2")

# Persistent ChromaDB
client = chromadb.PersistentClient(path=str(DB_DIR))

collection = client.get_or_create_collection(
    name="zepto_policies",
    metadata={"hnsw:space": "cosine"}
)


def chunk_text(text, chunk_size=500, overlap=80):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


documents = []
ids = []
metadatas = []

doc_files = sorted(DOCS_DIR.glob("*.txt"))

for file in doc_files:
    text = file.read_text(encoding="utf-8")
    chunks = chunk_text(text)

    for i, chunk in enumerate(chunks):
        documents.append(chunk)
        ids.append(f"{file.stem}_chunk_{i}")
        metadatas.append({
            "source": file.name,
            "chunk": i
        })


embeddings = model.encode(documents).tolist()

collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)

print("Documents found:", len(doc_files))
print("Chunks created:", len(documents))
print("Chunks stored in ChromaDB:", collection.count())
print("ChromaDB path:", DB_DIR)

Writing ingest.py


In [9]:
!python ingest.py

modules.json: 100% 349/349 [00:00<00:00, 877kB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 406kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 13.5MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 86.0kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.06MB/s]

model.safetensors: downloading bytes:  94% 85.0M/90.9M [00:04<00:00, 16.4MB/s, 6.42MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:05<00:00, 15.7MB/s, 6.46MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:05<00:00, 16.8MB/s, 8.02MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 4158.53it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.18MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 7.28MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 21.2MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 383kB/s]
config.json: 100% 190/190 [00:00<00:00, 466kB/s]
Documents found: 8
Chunks created: 15
Chunks stored in ChromaDB: 15
ChromaDB path: /content/drive/MyDri

In [10]:
%%writefile prompt.py

SYSTEM_PROMPT = """
You are a Zepto customer support assistant.

Your job is to answer customer questions using ONLY the policy
information provided in the retrieved context.

Rules:
1. Use only the retrieved context to answer.
2. Do not make up information.
3. If the answer is not available in the context, clearly say that
   the information is not available in the provided policy documents.
4. Keep answers clear, concise, and helpful.
5. When possible, explain the relevant policy in simple language.
"""


def build_prompt(question, context):
    return f"""
{SYSTEM_PROMPT}

Retrieved Policy Context:
{context}

Customer Question:
{question}

Answer:
"""

Writing prompt.py


In [11]:
!cat prompt.py


SYSTEM_PROMPT = """
You are a Zepto customer support assistant.

Your job is to answer customer questions using ONLY the policy
information provided in the retrieved context.

Rules:
1. Use only the retrieved context to answer.
2. Do not make up information.
3. If the answer is not available in the context, clearly say that
   the information is not available in the provided policy documents.
4. Keep answers clear, concise, and helpful.
5. When possible, explain the relevant policy in simple language.
"""


def build_prompt(question, context):
    return f"""
{SYSTEM_PROMPT}

Retrieved Policy Context:
{context}

Customer Question:
{question}

Answer:
"""


In [12]:
%%writefile prompt.py

PROMPT_TEMPLATE = """
ROLE:
You are a Zepto customer support assistant. Your job is to answer
customer questions accurately using Zepto policy information.

CONTEXT:
Use only the following retrieved policy context:

{context}

TASK:
Answer the customer's question based only on the provided context.

Customer Question:
{question}

FORMAT:
Provide a clear and concise answer in plain text.
Do not add unsupported details.

LENGTH:
Keep the answer short, preferably 2 to 4 sentences.

NEGATIVE CONSTRAINT:
Do not answer using information that is not present in the provided
policy context. Do not make up policies, fees, timelines, or rules.

FEW-SHOT EXAMPLE:

Context:
Standard delivery is free on orders over INR 149.
Orders below INR 149 incur a flat INR 25 delivery fee.

Question:
What is the delivery fee for an order below INR 149?

Answer:
Orders below INR 149 have a flat delivery fee of INR 25.

NOW ANSWER THE ACTUAL QUESTION:

Context:
{context}

Question:
{question}

Answer:
"""


def build_prompt(question, context):
    return PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

Overwriting prompt.py


In [13]:
!cat prompt.py


PROMPT_TEMPLATE = """
ROLE:
You are a Zepto customer support assistant. Your job is to answer
customer questions accurately using Zepto policy information.

CONTEXT:
Use only the following retrieved policy context:

{context}

TASK:
Answer the customer's question based only on the provided context.

Customer Question:
{question}

FORMAT:
Provide a clear and concise answer in plain text.
Do not add unsupported details.

LENGTH:
Keep the answer short, preferably 2 to 4 sentences.

NEGATIVE CONSTRAINT:
Do not answer using information that is not present in the provided
policy context. Do not make up policies, fees, timelines, or rules.

FEW-SHOT EXAMPLE:

Context:
Standard delivery is free on orders over INR 149.
Orders below INR 149 incur a flat INR 25 delivery fee.

Question:
What is the delivery fee for an order below INR 149?

Answer:
Orders below INR 149 have a flat delivery fee of INR 25.

NOW ANSWER THE ACTUAL QUESTION:

Context:
{context}

Question:
{question}

Answer:
"""


def 

In [14]:
!pip install -q langgraph

In [15]:
%%writefile graph.py

import os
import json
from pathlib import Path
from typing import TypedDict, List

import chromadb
from sentence_transformers import SentenceTransformer
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END

from prompt import build_prompt


# --------------------------------------------------
# Configuration
# --------------------------------------------------

BASE_DIR = Path(__file__).parent
DB_DIR = BASE_DIR / "chroma_db"

# Default graded mode = MOCK
MOCK_LLM = os.getenv("MOCK_LLM", "1") != "0"

POLICY_KEYWORDS = [
    "delivery",
    "return",
    "refund",
    "membership",
    "tracking",
    "cancel",
    "gift card",
    "support hours",
]


# --------------------------------------------------
# Pydantic output schema
# --------------------------------------------------

class AnswerResponse(BaseModel):
    answer: str
    sources: List[str]
    confidence: float = Field(ge=0.0, le=1.0)


# --------------------------------------------------
# LangGraph state
# --------------------------------------------------

class AssistantState(TypedDict, total=False):
    query: str
    intent: str
    answer: str
    sources: List[str]
    confidence: float
    retrieved_context: str


# --------------------------------------------------
# Load embedding model + ChromaDB
# --------------------------------------------------

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path=str(DB_DIR))

collection = client.get_collection(
    name="zepto_policies"
)


# --------------------------------------------------
# Optional real LLM helper
# Used only when MOCK_LLM=0
# --------------------------------------------------

def call_real_llm(prompt: str) -> str:
    """
    Optional real-LLM extension.

    To use:
    1. Set MOCK_LLM=0
    2. Install groq
    3. Set GROQ_API_KEY
    """

    try:
        from groq import Groq
    except ImportError:
        raise RuntimeError(
            "Groq is not installed. Run: pip install groq"
        )

    api_key = os.getenv("GROQ_API_KEY")

    if not api_key:
        raise RuntimeError(
            "GROQ_API_KEY is required when MOCK_LLM=0"
        )

    groq_client = Groq(api_key=api_key)

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()


def generate_validated_real_answer(prompt, sources):
    """
    Retry up to 2 additional times if the LLM output
    does not match the required JSON schema.
    """

    current_prompt = prompt

    for attempt in range(3):

        raw_output = call_real_llm(current_prompt)

        try:
            result = AnswerResponse.model_validate_json(raw_output)
            return result

        except Exception:

            current_prompt += """

IMPORTANT:
Your previous response did not match the required JSON schema.

Return ONLY valid JSON in exactly this structure:

{
  "answer": "your answer",
  "sources": ["source_id"],
  "confidence": 0.9
}

Do not include Markdown or any text outside the JSON.
"""

    return AnswerResponse(
        answer="ERROR: Real LLM failed to produce valid structured output.",
        sources=sources,
        confidence=0.0
    )


# --------------------------------------------------
# Node 1: classify_intent
# --------------------------------------------------

def classify_intent(state: AssistantState):

    query = state["query"]

    if MOCK_LLM:

        query_lower = query.lower()

        if any(keyword in query_lower for keyword in POLICY_KEYWORDS):
            intent = "policy_question"
        else:
            intent = "general_question"

    else:

        classification_prompt = f"""
Classify the following query into exactly one category:

policy_question
general_question

A policy question is about Zepto delivery, returns, refunds,
membership, tracking, cancellation, gift cards or support hours.

Query:
{query}

Return only the category name.
"""

        result = call_real_llm(classification_prompt).lower()

        if "policy_question" in result:
            intent = "policy_question"
        else:
            intent = "general_question"

    return {
        "intent": intent
    }


# --------------------------------------------------
# Node 2: retrieve_and_answer
# --------------------------------------------------

def retrieve_and_answer(state: AssistantState):

    query = state["query"]

    # Embed user query
    query_embedding = embedding_model.encode(
        [query]
    ).tolist()

    # Retrieve top 3 chunks
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    documents = results["documents"][0]
    ids = results["ids"][0]

    context = "\n\n".join(documents)

    if MOCK_LLM:

        # Required deterministic mock output
        top_chunk = documents[0]

        top_chunk_snippet = top_chunk[:200].replace(
            "\n", " "
        )

        answer = (
            f"Based on the retrieved context: "
            f"{top_chunk_snippet}"
        )

        response = AnswerResponse(
            answer=answer,
            sources=ids,
            confidence=1.0
        )

    else:

        prompt = build_prompt(
            question=query,
            context=context
        )

        prompt += f"""

Return the final response as valid JSON with this schema:

{{
    "answer": "string",
    "sources": {json.dumps(ids)},
    "confidence": 0.0
}}
"""

        response = generate_validated_real_answer(
            prompt,
            ids
        )

    return {
        "answer": response.answer,
        "sources": response.sources,
        "confidence": response.confidence,
        "retrieved_context": context
    }


# --------------------------------------------------
# Node 3: direct_answer
# --------------------------------------------------

def direct_answer(state: AssistantState):

    query = state["query"]

    if MOCK_LLM:

        response = AnswerResponse(
            answer=(
                "I can only answer questions about "
                "Zepto policies right now."
            ),
            sources=[],
            confidence=1.0
        )

    else:

        prompt = f"""
You are a Zepto customer support assistant.

The following question does not require policy retrieval.

Question:
{query}

Return ONLY JSON:

{{
    "answer": "short answer",
    "sources": [],
    "confidence": 0.9
}}
"""

        response = generate_validated_real_answer(
            prompt,
            []
        )

    return {
        "answer": response.answer,
        "sources": response.sources,
        "confidence": response.confidence
    }


# --------------------------------------------------
# Conditional routing
# --------------------------------------------------

def route_intent(state: AssistantState):

    return state["intent"]


# --------------------------------------------------
# Build LangGraph
# --------------------------------------------------

builder = StateGraph(AssistantState)

builder.add_node(
    "classify_intent",
    classify_intent
)

builder.add_node(
    "retrieve_and_answer",
    retrieve_and_answer
)

builder.add_node(
    "direct_answer",
    direct_answer
)

builder.add_edge(
    START,
    "classify_intent"
)

builder.add_conditional_edges(
    "classify_intent",
    route_intent,
    {
        "policy_question": "retrieve_and_answer",
        "general_question": "direct_answer"
    }
)

builder.add_edge(
    "retrieve_and_answer",
    END
)

builder.add_edge(
    "direct_answer",
    END
)

graph = builder.compile()


# --------------------------------------------------
# Helper used later by FastAPI
# --------------------------------------------------

def ask_question(query: str) -> AnswerResponse:

    result = graph.invoke({
        "query": query
    })

    return AnswerResponse(
        answer=result["answer"],
        sources=result["sources"],
        confidence=result["confidence"]
    )

Writing graph.py


In [16]:
from graph import graph

result = graph.invoke({
    "query": "What is Zepto's delivery policy?"
})

print(result)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'query': "What is Zepto's delivery policy?", 'intent': 'policy_question', 'answer': "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del", 'sources': ['doc_01_chunk_0', 'doc_03_chunk_0', 'doc_04_chunk_0'], 'confidence': 1.0, 'retrieved_context': "Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin cod\n\nZepto offers three account tiers: Basic (free, default tier, standa

In [17]:
from graph import graph
import json

result = graph.invoke({
    "query": "What is Zepto's delivery policy?"
})

print(json.dumps(result, indent=2))

{
  "query": "What is Zepto's delivery policy?",
  "intent": "policy_question",
  "answer": "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del",
  "sources": [
    "doc_01_chunk_0",
    "doc_03_chunk_0",
    "doc_04_chunk_0"
  ],
  "confidence": 1.0,
  "retrieved_context": "Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin cod\n\nZepto offers three account tiers: Basi

In [18]:
result = graph.invoke({
    "query": "Who is the president of India?"
})

print(json.dumps(result, indent=2))

{
  "query": "Who is the president of India?",
  "intent": "general_question",
  "answer": "I can only answer questions about Zepto policies right now.",
  "sources": [],
  "confidence": 1.0
}


In [19]:
!pip install -q fastapi uvicorn httpx

In [20]:
%%writefile main.py

from fastapi import FastAPI
from pydantic import BaseModel

from graph import ask_question, AnswerResponse


app = FastAPI(
    title="Zepto Support Assistant",
    description="RAG-based Zepto policy support assistant",
    version="1.0.0"
)


class AskRequest(BaseModel):
    query: str


@app.get("/")
def root():
    return {
        "message": "Zepto Support Assistant API is running"
    }


@app.post("/ask", response_model=AnswerResponse)
def ask(request: AskRequest):
    return ask_question(request.query)

Writing main.py


In [21]:
from fastapi.testclient import TestClient
from main import app

client = TestClient(app)

response = client.post(
    "/ask",
    json={"query": "What is Zepto's delivery policy?"}
)

print(response.status_code)
print(response.json())

200
{'answer': "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del", 'sources': ['doc_01_chunk_0', 'doc_03_chunk_0', 'doc_04_chunk_0'], 'confidence': 1.0}


In [22]:
response = client.post(
    "/ask",
    json={"query": "Who is the president of India?"}
)

print(response.status_code)
print(response.json())

200
{'answer': 'I can only answer questions about Zepto policies right now.', 'sources': [], 'confidence': 1.0}


In [23]:
import subprocess
import time

server = subprocess.Popen(
    [
        "python", "-m", "uvicorn",
        "main:app",
        "--host", "127.0.0.1",
        "--port", "8000"
    ]
)

time.sleep(8)

print("Uvicorn server started")
print("PID:", server.pid)

Uvicorn server started
PID: 8060


In [24]:
import requests
import json

response = requests.post(
    "http://127.0.0.1:8000/ask",
    json={
        "query": "What is Zepto's delivery policy?"
    }
)

print("Status:", response.status_code)
print(json.dumps(response.json(), indent=2))

Status: 200
{
  "answer": "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del",
  "sources": [
    "doc_01_chunk_0",
    "doc_03_chunk_0",
    "doc_04_chunk_0"
  ],
  "confidence": 1.0
}


In [25]:
response = requests.post(
    "http://127.0.0.1:8000/ask",
    json={
        "query": "Who is the president of India?"
    }
)

print("Status:", response.status_code)
print(json.dumps(response.json(), indent=2))

Status: 200
{
  "answer": "I can only answer questions about Zepto policies right now.",
  "sources": [],
  "confidence": 1.0
}


In [26]:
%%writefile requirements.txt
fastapi
uvicorn
chromadb
sentence-transformers
langgraph
pydantic
requests
groq

Writing requirements.txt


In [27]:
%%writefile Dockerfile

FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

# Cache the embedding model inside the Docker image
RUN python -c "from sentence_transformers import SentenceTransformer; SentenceTransformer('all-MiniLM-L6-v2')"

COPY . .

# Build/populate ChromaDB from the 8 policy documents
RUN python ingest.py

ENV MOCK_LLM=1

EXPOSE 7860

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"]

Writing Dockerfile


In [28]:
%%writefile README.md

# Zepto Support Assistant

## Overview

This module implements a small Retrieval-Augmented Generation (RAG)
support assistant for Zepto policies.

The system uses:

- Sentence Transformers for embeddings
- ChromaDB for vector storage
- LangGraph for query routing
- Pydantic for structured responses
- FastAPI for the REST API
- Uvicorn for serving the application

The default graded implementation uses `MOCK_LLM=1`, so no external
LLM API or API key is required.

---

## Project Structure

```text
support_assistant/
├── docs/
│   ├── doc_01.txt
│   ├── doc_02.txt
│   ├── doc_03.txt
│   ├── doc_04.txt
│   ├── doc_05.txt
│   ├── doc_06.txt
│   ├── doc_07.txt
│   └── doc_08.txt
├── chroma_db/
├── ingest.py
├── prompt.py
├── graph.py
├── main.py
├── requirements.txt
├── Dockerfile
└── README.md

Writing README.md


In [29]:
!find . -maxdepth 2 -type f | sort

./chroma_db/chroma.sqlite3
./Dockerfile
./docs/doc_01.txt
./docs/doc_02.txt
./docs/doc_03.txt
./docs/doc_04.txt
./docs/doc_05.txt
./docs/doc_06.txt
./docs/doc_07.txt
./docs/doc_08.txt
./graph.py
./ingest.py
./main.py
./module3_builder.ipynb
./prompt.py
./__pycache__/graph.cpython-313.pyc
./__pycache__/main.cpython-313.pyc
./__pycache__/prompt.cpython-313.pyc
./README.md
./requirements.txt


In [30]:
from graph import MOCK_LLM
print("MOCK_LLM:", MOCK_LLM)

MOCK_LLM: True


In [31]:
!rm -rf __pycache__

In [32]:
!find . -maxdepth 2 -type f | sort

./chroma_db/chroma.sqlite3
./Dockerfile
./docs/doc_01.txt
./docs/doc_02.txt
./docs/doc_03.txt
./docs/doc_04.txt
./docs/doc_05.txt
./docs/doc_06.txt
./docs/doc_07.txt
./docs/doc_08.txt
./graph.py
./ingest.py
./main.py
./module3_builder.ipynb
./prompt.py
./README.md
./requirements.txt


In [33]:
%cd /content/drive/MyDrive/zepto-data-ai-platform
!pwd
!ls

/content/drive/MyDrive/zepto-data-ai-platform
/content/drive/MyDrive/zepto-data-ai-platform
analytics  data_pipeline  support_assistant


In [34]:
%%writefile README.md

# Zepto Data & AI Platform

## Overview

This repository contains a complete data and AI project organized into three independent modules:

1. **Data Pipeline** — data ingestion, cleaning, validation, transformation, and storage.
2. **Analytics** — exploratory data analysis and machine-learning modeling.
3. **Support Assistant** — a RAG-based Zepto policy assistant using embeddings, ChromaDB, LangGraph, and FastAPI.

All three modules are maintained inside one repository as required.

---

## Repository Structure

```text
zepto-data-ai-platform/
├── data_pipeline/
├── analytics/
├── support_assistant/
└── README.md

Writing README.md


In [36]:
!ls

books_cleaned.csv  module1_data_pipeline.ipynb	README.md
books.db	   query_outputs.txt		requirements.txt


In [38]:
!pwd

/root


In [39]:
%cd /content/drive/MyDrive/zepto-data-ai-platform

/content/drive/MyDrive/zepto-data-ai-platform


In [40]:
!pwd

/content/drive/MyDrive/zepto-data-ai-platform


In [41]:
!ls

analytics  data_pipeline  README.md  support_assistant


In [42]:
!pwd
!git status

/content/drive/MyDrive/zepto-data-ai-platform
fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [43]:
!git init -b main

Initialized empty Git repository in /content/drive/MyDrive/zepto-data-ai-platform/.git/


In [44]:
!git config user.name "KaushikAnde"
!git config user.email "andekaushik@gmail.com"

In [45]:
%%writefile .gitignore
__pycache__/
*.pyc
.ipynb_checkpoints/
.DS_Store
.env

# Generated vector database
support_assistant/chroma_db/

Writing .gitignore


In [46]:
!git status --short

?? .gitignore
?? README.md
?? analytics/
?? data_pipeline/
?? support_assistant/
